In [ ]:
### Régression Logistique fonctionnel ###




# Entraînement du classificateur
lr = LogisticRegression()
lr.fit(X_test,y_test)

#construction de y_pred
y_pred=lr.predict(X_test)

#Regardons la distribution de probabilité qu'une transaction soit frauduleuse 
y_pred_proba = lr.predict_proba(X_test)

# Métriques au seuil de 50%
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
fpr, tpr, ths = roc_curve(y_test, y_pred_proba[:,1])
roc_auc = auc(fpr, tpr)
accuracy=accuracy_score(y_test,y_pred)

#Matrice de confusion au seuil de 50%
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred,labels = [1,0])


#On peut tracer un histogramme de ces probas
pd.Series(y_pred_proba[:,1]).hist()


"""
fpr : FP rate
tpr : TP rate
ths : threshold

"""

#La fonction roc_curve permet de tracer la courbe ROC 
# à partir d'un vecteur de probabilités de la classe positive (ici (fraudulent=1))
fpr, tpr, ths = roc_curve(y_test, y_pred_proba[:,1])
auc_score = auc(fpr,tpr)
plt.plot(fpr,tpr,label="AUC Score:" + str(auc_score))
plt.xlabel('FALSE POSITIVE rate',fontsize='15')
plt.ylabel('TRUE POSITIVE rate',fontsize='15')
plt.legend(loc='best')

#Recherche d'un seuil (threshold) idéal 
"""
On souhaiterait maximiser le taux de détection d'une transaction frauduleuse
tout en minimisant le taux de transaction non frauduleuses classées comme tel

On va pour cela agir sur le seuil de classification (threshold)

Dans un premier temps, on cherche un seuil qui maximise le recall (sensitivity 
i.e le TP rate) tout en minimisant le FP rate. Néanmoins, le seuil obtenu n'est pas 
le plus optimal selon les situations. 

On va donc dans un second temps utiliser l'indice de Youden qui va nous donner un 
autre seuil de façon optimale selon l'indice de Youden. 
"""

#Recherche seuil sans indice de Youden

index_max= np.where(tpr==np.max(tpr))[0] #indices pour lesquelle tpr est max
threshold_index=np.max(index_max)
threshold= ths[threshold_index]

#On peut le voir graphiquement
plt.plot(ths,tpr)
plt.xlabel('threshold')
plt.ylabel('TRUE POSITIVE rate')

#On obtient donc de nouvelles valeurs prédites ajustées
from sklearn.preprocessing import binarize
y_pred_th= binarize(y_pred_proba, threshold=threshold)
confusion_mat=confusion_matrix(y_test, y_pred_th[:,1],labels=[1,0])
    
#Les métriques associées
recall=tpr[threshold_index]
precision=(confusion_mat[0,0])/(confusion_mat[0,0] + confusion_mat[1,0])
specificity=1-fpr[threshold_index]
f1_score=2*precision*recall/(precision+recall)

#Recherche d'un seuil avec l'indice de Youden
"""
On va cette fois-ci maximiser la quantité : recall + specificty - 1 
On en tirera un seuil optimal noté threshold_y
On s'intéressera aux métrques associées à ce seuil afin de les comparer à
celles sans l'indice de Youden

"""
youden= tpr - fpr
index_max_y= np.where(youden==np.max(youden))
threshold_y_index= np.max(index_max_y)
threshold_y=ths[threshold_y_index]


#On obtient de même de nouvelles valeurs prédites ajustées
y_pred_th_y= binarize(y_pred_proba, threshold=threshold_y)
confusion_mat_y = confusion_matrix(y_test, y_pred_th_y[:,1],labels=[1,0])

#Les métriques associées avec Youden
recall_y=tpr[threshold_y_index]
precision_y=(confusion_mat_y[0,0])/(confusion_mat_y[0,0] + confusion_mat_y[1,0])
specificity_y=1-fpr[threshold_y_index]
f1_score_y=2*precision_y*recall_y/(precision_y+recall_y)

#Affichage des métriques utiles
print("accuracy:" +str(accuracy))
print("AUC: "+ str(auc_score))
print("threshold: " + str(threshold) )
print("recall (sensitivity) : " + str(recall) )
print("precision :" + str(precision))
print("specificity : " + str(specificity) )
print("f1-score avec Youden: " + str(f1_score))
print("threshold avec Youden: " + str(threshold_y) )
print("recall (sensitivity) avec Youden : " + str(recall_y) )
print("precision avec Youden:" + str(precision_y))
print("specificity avec Youden: " + str(specificity_y) )
print("f1-score avec Youden : " + str(f1_score_y))
print("Indice de Youden: " + str(youden[threshold_y_index]))

# Tracé de la courbe ROC
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.xlabel('Taux de Faux Positifs')
plt.ylabel('Taux de Vrais Positifs')
plt.title('Courbe ROC (AUC = %.3f)' % roc_auc)
plt.show()


